# Milestone 0: Quadratic Pipeline Verification

Verify the mathematical infrastructure:
- Schur complement matches direct inversion
- Pole sums for Delta and Sigma
- Gateway G_dd = 1/(iw + mu - eps_d - sigma_inf - Delta - Sigma_h)
- Matsubara sum correlators match exact Fermi occupancies
- Bethe lattice GF: causality, high-freq tail, self-consistency

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from dmft.matsubara import matsubara_frequencies, fermi_function, pole_matsubara_sum
from dmft.schur import schur_complement_diag, block_greens_functions
from dmft.greens_function import hybridization, self_energy_poles
from dmft.lattice import bethe_local_gf
from dmft.gateway import gateway_greens_functions, gateway_correlators, gateway_onebody_matrix

## 1. Schur complement vs direct inversion

In [ ]:
# Build a (1+M) inverse resolvent and compare Schur vs direct
M = 3
iw_test = 1j * 1.57
mu, eps_d = 0.5, 0.0
V = np.array([0.2, 0.4, 0.3])
e = np.array([-0.5, 0.0, 0.7])

# Full matrix
dim = 1 + M
G_inv = np.zeros((dim, dim), dtype=complex)
G_inv[0, 0] = iw_test + mu - eps_d
for l in range(M):
    G_inv[0, l+1] = -np.conj(V[l])
    G_inv[l+1, 0] = -V[l]
    G_inv[l+1, l+1] = iw_test - e[l]

G_full = np.linalg.inv(G_inv)
G_dd_direct = G_full[0, 0]

# Schur complement
A = iw_test + mu - eps_d
S = schur_complement_diag(A, -np.conj(V), -V, iw_test - e)
G_dd_schur = 1.0 / S

print(f'G_dd (direct):     {G_dd_direct:.10f}')
print(f'G_dd (Schur):      {G_dd_schur:.10f}')
print(f'Difference:        {abs(G_dd_direct - G_dd_schur):.2e}')

## 2. Bethe lattice Green's function

In [ ]:
beta = 50.0
n_w = 1024
wn = matsubara_frequencies(n_w, beta)
iw = 1j * wn
t = 0.5

# Non-interacting G_loc on Bethe lattice
G_loc = bethe_local_gf(iw, 0.0, 0.0, np.zeros(n_w, dtype=complex), t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(wn[:50], G_loc[:50].real, 'b.-', ms=3)
ax1.set_xlabel(r'$\omega_n$'); ax1.set_ylabel(r'Re $G_{loc}$')
ax1.set_title('Non-interacting Bethe lattice')

ax2.plot(wn[:50], G_loc[:50].imag, 'r.-', ms=3)
ax2.set_xlabel(r'$\omega_n$'); ax2.set_ylabel(r'Im $G_{loc}$')
ax2.set_title('Causal: Im G < 0')
plt.tight_layout(); plt.show()

# Verify self-consistency: t^2 G^2 - iw*G + 1 = 0
residual = t**2 * G_loc**2 - iw * G_loc + 1.0
print(f'Self-consistency residual: {np.max(np.abs(residual)):.2e}')
print(f'High-freq tail: G[-1] = {G_loc[-1]:.6f}, 1/iw[-1] = {1/iw[-1]:.6f}')

## 3. Gateway correlators: exact diagonalization vs Matsubara sums

In [ ]:
V_gw = np.array([0.3, 0.5])
eps_gw = np.array([-0.4, 0.4])
W_gw = np.array([0.2, 0.4])
eta_gw = np.array([-0.5, 0.5])
mu_gw, eps_d_gw, sigma_inf_gw = 1.0, 0.0, 1.0

# Exact correlators via diagonalization
corr = gateway_correlators(mu_gw, eps_d_gw, sigma_inf_gw,
                           V_gw, eps_gw, W_gw, eta_gw, beta)

# Verify via direct diagonalization of K
K = gateway_onebody_matrix(mu_gw, eps_d_gw, sigma_inf_gw,
                           V_gw, eps_gw, W_gw, eta_gw)
eigvals, U_mat = np.linalg.eigh(K)
f_e = fermi_function(eigvals, beta)
f_matrix = U_mat @ np.diag(f_e) @ U_mat.T

print('Gateway correlators (exact diag):')
print(f'  <g_l^dag g_l>: {corr["gg"]}')
print(f'  <h_l^dag h_l>: {corr["hh"]}')
print(f'  <d^dag g_l>:   {corr["dg"]}')
print(f'  <d^dag h_l>:   {corr["dh"]}')
print(f'\nAll occupancies in [0,1]: {np.all(corr["gg"] >= 0) and np.all(corr["gg"] <= 1)}')
print(f'K eigenvalues: {eigvals}')
print(f'Fermi occupancies: {f_e}')

## 4. Pole sum identity verification

Key identity: $(1/\beta) \sum_n 1/(i\omega_n - x) = -f(x)$

In [ ]:
x_values = np.linspace(-2, 2, 20)
exact = -fermi_function(x_values, beta)
pole_sum = np.array([pole_matsubara_sum(np.array([1.0]), np.array([x]), beta) for x in x_values])

plt.figure(figsize=(6, 4))
plt.plot(x_values, exact, 'b-', label='$-f(x)$ exact')
plt.plot(x_values, pole_sum, 'ro', ms=4, label='Pole Matsubara sum')
plt.xlabel('$x$'); plt.ylabel('$(1/\\beta) \\sum_n 1/(i\\omega_n - x)$')
plt.legend(); plt.title(f'Pole sum identity ($\\beta={beta}$)')
plt.tight_layout(); plt.show()
print(f'Max error: {np.max(np.abs(exact - pole_sum)):.2e}')